# 🚗 Automatic Number Plate Recognition (ANPR) — YOLOv9 + EasyOCR

**Rebuilt from:** `Automatic-number-plate-recognition-ANPR-with-Yolov9-and-EasyOCR` (original repo)

This notebook is a senior-CV-engineer rewrite of the original project. It is **self-contained and runnable top-to-bottom** in Google Colab (GPU runtime recommended). No web app, no tunnels, no browser networking to fight with — just:

1. Upload a video (Colab's built-in file picker, one cell)
2. Run the pipeline cell
3. Get back an **annotated output video** (boxes + track IDs + plate text), shown right in the notebook, **and** a **plain-text report** listing every distinct vehicle that passed, with its plate number, confidence, and timestamps

---

## 🔍 Code review of the original repo (what I found)

| # | Issue in original `anpr.py` / notebook | Why it matters |
|---|---|---|
| 1 | No object **tracking** — every frame is treated independently | The same car gets OCR'd 100s of times with no way to say "this is one vehicle." Impossible to build a "vehicles passed" log, and EasyOCR (slow) runs on every single frame/box. |
| 2 | OCR acceptance rule: `len(results)==1 or (len(text)>6 and conf>0.2)` | Rejects many real plates (lots of plate formats are 5–7 chars) while blindly accepting *any* text when EasyOCR happens to return exactly one guess, regardless of confidence. |
| 3 | `easyocr.Reader(['en'], gpu=True)` hardcoded | Crashes immediately on a CPU-only runtime. |
| 4 | `cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)` on a frame read by OpenCV | OpenCV frames are BGR, not RGB — the constant name is misleading (works "by accident" for grayscale, but it's a red flag in review). |
| 5 | No bounds/empty-crop checking before `cvtColor` | A box touching the frame edge produces a zero-size crop → hard crash. |
| 6 | Output video written with `mp4v` fourcc | Frequently **won't play in-browser** (Chrome needs H.264). Original script has no re-encode step. |
| 7 | Entire pipeline is a CLI script bolted onto `detect.py` | 300+ lines duplicated from YOLOv9's own `detect.py`, tightly coupled to the repo's internal APIs (`DetectMultiBackend`, `LoadImages`, ...) — brittle, hard to extend, no way to reuse programmatically. |
| 8 | Only produces console logs (`LOGGER.info`) | No structured output (CSV/table) — nothing you can act on afterwards. |
| 9 | Zero results until you train | The notebook forces you through a full 25-epoch training run (with a personal Roboflow account) before you can test *anything*. |

## 🛠️ What this rewrite does differently

- **Migrates to the `ultralytics` API.** YOLOv9 (and GELAN) are natively supported by `ultralytics>=8.1`, which replaces ~300 lines of repo-specific boilerplate with a few clean calls, and gives us **built-in multi-object tracking (ByteTrack)** for free via `model.track(...)`.
- **Tracking-first design.** Every detection gets a persistent track ID. We OCR each track every few frames (not every frame) and keep the **highest-confidence plate reading per vehicle** — this is what makes a real "track record" possible.
- **Two-tier plate cropping.** If you plug in a plate-specific detector (trained in the optional Section 2 below), we crop the plate box directly. If you're just running the generic COCO-pretrained `yolov9c.pt` (no training needed), we fall back to a heuristic lower-bumper crop of each vehicle box so the pipeline **works immediately**, and upgrades in accuracy the moment you drop in a trained plate model.
- **Robust OCR gating.** Plate text is cleaned (alnum, uppercase) and only accepted if it looks like a plausible plate (length 4–10) and beats the previous best confidence for that track.
- **Browser-safe output.** Output video is re-encoded with `ffmpeg` (H.264) so it plays reliably, including inline in this notebook.
- **Structured log.** Results land in a plain-text report: one line per vehicle — track ID, class, plate text, confidence, first/last seen (seconds) — plus a de-duplicated list of plate numbers actually read.
- **No Gradio.** Dropped entirely for this version — a direct function call + inline video/report display is simpler to test and has no networking failure modes.

> ⚠️ **Note on weights:** the original zip does not ship a trained plate-detector `.pt` file (only code + a sample `car.mp4`). Section 2 (optional) trains one from the same Roboflow dataset the original notebook used. If you skip it, Section 3 automatically falls back to the generic COCO-pretrained `yolov9c.pt` so the pipeline still runs end-to-end today — just with the heuristic crop instead of a true plate box.

## 0. Check GPU
Go to `Runtime → Change runtime type → GPU` first if this fails or shows no GPU.

In [ ]:
!nvidia-smi

## 1. Install dependencies
One clean install — no repo cloning, no `requirements.txt` from a third-party fork.

In [ ]:
!pip install -q -U "ultralytics>=8.1.0" easyocr pandas opencv-python-headless lapx roboflow
!apt-get -qq install -y ffmpeg > /dev/null
print("✅ Dependencies installed")

In [ ]:
import os, glob, re, time
import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO
import easyocr

HOME = os.getcwd()
print("HOME:", HOME)
print("CUDA available:", torch.cuda.is_available())

## 2. (Optional) Train a license-plate detector

Skip this whole section if you already have a trained `best.pt`, or if you just want to try the pipeline right now with the generic COCO-pretrained model (Section 3 will fall back automatically) — though as you just saw, the generic fallback only *guesses* at the plate's location, so training this once is strongly worth it.

This trains on the **same Roboflow ANPR dataset** the original notebook used (workspace `arvind-kumar-wjygd`, project `anpr2-syxl7`, version 8 — verified to match the original notebook exactly), through the modern `ultralytics` training API (no cloning the `yolov9` repo, no `train.py` CLI).

**To run it, you need a free Roboflow API key:**
1. Sign up at [roboflow.com](https://roboflow.com) (free, no card required).
2. Go to **Settings → Roboflow API** (or your account menu → API Keys) and copy your key.
3. Paste it into `ROBOFLOW_API_KEY` below, or leave it blank to use the interactive `roboflow.login()` browser flow instead.

Set `TRAIN_NEW_MODEL = True` to run it (~15–20 min on a T4 GPU for 25 epochs).

In [ ]:
TRAIN_NEW_MODEL = False   # <-- flip to True to train your own plate detector
ROBOFLOW_API_KEY = ""     # <-- paste your free Roboflow API key here (recommended), or leave blank for interactive login

if TRAIN_NEW_MODEL:
    import roboflow

    if ROBOFLOW_API_KEY:
        rf = roboflow.Roboflow(api_key=ROBOFLOW_API_KEY)
    else:
        roboflow.login()   # opens a browser tab; log in and it hands the key back automatically
        rf = roboflow.Roboflow()

    project = rf.workspace("arvind-kumar-wjygd").project("anpr2-syxl7")
    version = project.version(8)
    dataset = version.download("yolov9")

    model = YOLO("yolov9c.pt")  # start from COCO-pretrained weights
    train_results = model.train(
        data=f"{dataset.location}/data.yaml",
        epochs=25,
        imgsz=640,
        batch=16,
        device=0,
        name="anpr_plate_detector",
    )
    print("Training complete. Best weights at:",
          glob.glob(f"{HOME}/runs/detect/anpr_plate_detector*/weights/best.pt"))
else:
    print("Skipping training — using fallback / uploaded weights in Section 3.")

## 3. Resolve model weights

Priority order:
1. A `best.pt` produced by Section 2 (if you trained one)
2. A weights file you upload right now (optional — run the cell, click "Choose Files" if you have a `best.pt` from a previous run, or press Cancel/skip to use the fallback)
3. Fallback: generic COCO-pretrained `yolov9c.pt`, auto-downloaded by `ultralytics` (detects vehicles: car/truck/bus/motorcycle — plate text will use the heuristic crop, see notes above)

In [ ]:
trained_candidates = sorted(
    glob.glob(f"{HOME}/runs/detect/*/weights/best.pt"),
    key=os.path.getmtime, reverse=True
)

DEFAULT_WEIGHTS = trained_candidates[0] if trained_candidates else "yolov9c.pt"

if trained_candidates:
    print(f"✅ Found a trained plate detector: {DEFAULT_WEIGHTS}")
else:
    print("ℹ️ No trained plate detector found — defaulting to generic 'yolov9c.pt' (COCO vehicle classes).")
    print("   You can still manually point DEFAULT_WEIGHTS to your own best.pt below.")

# Trigger the ultralytics auto-download now so the first run isn't slow
_ = YOLO(DEFAULT_WEIGHTS)
print("Default weights ready:", DEFAULT_WEIGHTS)

## 4. Core ANPR pipeline

- `read_plate()` — EasyOCR wrapper with cleanup + plausibility gating (fixes issue #2/#3 above)
- `get_plate_box()` — returns a tight box for a plate-specific model, or a heuristic lower-bumper-band box (not the whole vehicle) when running the generic vehicle detector — this is what gets drawn and cropped for OCR
- `_process_video_impl()` — runs `model.track()` (built-in ByteTrack, restricted to car/truck/bus/motorcycle classes on the generic model so people/bicycles/etc. are never boxed), skips vehicles too small/far away to plausibly read a plate, keeps the best plate reading per track ID, draws only the plate-region box, re-encodes with ffmpeg, and returns a text report

> ⚠️ With the generic COCO fallback model there's no real plate detector — the drawn box is a **heuristic estimate of where a plate usually sits** on a vehicle, not a true plate detection. For an actual tight plate box every time, train a real plate detector in Section 2 (or supply your own).

In [ ]:
_ocr_reader = None

def get_ocr_reader():
    global _ocr_reader
    if _ocr_reader is None:
        _ocr_reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
    return _ocr_reader


_ALNUM_RE = re.compile(r'[^A-Z0-9]')

def clean_plate_text(raw_text):
    return _ALNUM_RE.sub('', raw_text.upper())


def is_plausible_plate(text):
    return 4 <= len(text) <= 10


def read_plate(crop_bgr):
    """Run EasyOCR on a BGR crop, return (best_text, best_confidence)."""
    if crop_bgr is None or crop_bgr.size == 0:
        return "", 0.0

    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)

    h, w = gray.shape[:2]
    if 0 < w < 200:
        scale = 200 / w
        gray = cv2.resize(gray, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_CUBIC)

    reader = get_ocr_reader()
    try:
        results = reader.readtext(gray)
    except Exception:
        return "", 0.0

    best_text, best_conf = "", 0.0
    for (_, text, conf) in results:
        cleaned = clean_plate_text(text)
        if is_plausible_plate(cleaned) and conf > best_conf:
            best_text, best_conf = cleaned, conf
    return best_text, best_conf


def get_plate_box(xyxy, class_name, frame_shape):
    """Return the (x1,y1,x2,y2) region to treat as 'the plate'.
    Tight box for a real plate-class detection; heuristic lower-bumper
    band for a generic vehicle-class detection (no plate model loaded)."""
    x1, y1, x2, y2 = map(int, xyxy)
    x1, y1 = max(x1, 0), max(y1, 0)
    x2, y2 = min(x2, frame_shape[1]), min(y2, frame_shape[0])
    if x2 <= x1 or y2 <= y1:
        return None

    if 'plate' in class_name.lower():
        return (x1, y1, x2, y2)

    # Fallback for generic vehicle detectors: plates usually sit in the lower-center band
    h, w = y2 - y1, x2 - x1
    ny1, ny2 = y1 + int(h * 0.60), y2 - int(h * 0.08)
    nx1, nx2 = x1 + int(w * 0.28), x2 - int(w * 0.28)
    if ny2 <= ny1 or nx2 <= nx1:
        return (x1, y1, x2, y2)
    return (nx1, ny1, nx2, ny2)


def crop_region(frame, box):
    if box is None:
        return None
    x1, y1, x2, y2 = box
    return frame[y1:y2, x1:x2]

In [ ]:
CONF_THRES = 0.35        # detection confidence threshold (raised from 0.25 to cut noisy far-away boxes)
OCR_EVERY_N_FRAMES = 5   # run OCR this often per tracked vehicle (higher = faster, lower = more accurate)
MIN_VEHICLE_BOX_HEIGHT_FRAC = 0.14  # ignore vehicles shorter than 14% of frame height (too far away to read a plate anyway)

# COCO class ids for the generic yolov9c.pt fallback — restrict tracking to these so
# people/bicycles/parking meters/etc. never get boxed or logged as "vehicles".
VEHICLE_COCO_IDS = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}


def _process_video_impl(video_path, progress):
    weights_path = DEFAULT_WEIGHTS
    conf_thres = CONF_THRES
    ocr_interval = OCR_EVERY_N_FRAMES

    progress(0, desc="Loading model...")
    model = YOLO(weights_path)
    class_names = model.names
    # Anything other than the generic COCO fallback ("yolov9c.pt") is assumed to be a real
    # plate detector — trained in Section 2, or your own uploaded best.pt — regardless of
    # what its class is actually named in whatever dataset it was trained on.
    is_plate_model = os.path.basename(weights_path) != "yolov9c.pt"

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    if frame_w == 0 or frame_h == 0:
        raise RuntimeError(
            "OpenCV could not read this video (width/height came back 0). "
            "Try re-exporting it as a standard .mp4 (H.264) and upload again."
        )

    tmp_out = f"{HOME}/_tmp_output.mp4"
    writer = cv2.VideoWriter(tmp_out, cv2.VideoWriter_fourcc(*"mp4v"), fps, (frame_w, frame_h))

    track_data = {}
    frame_idx = 0

    progress(0, desc="Running detection + tracking...")
    track_kwargs = dict(
        source=video_path, stream=True, persist=True,
        conf=conf_thres, tracker="bytetrack.yaml", verbose=False,
    )
    if not is_plate_model:
        # Generic COCO model: only ever detect/track actual vehicle classes
        track_kwargs["classes"] = list(VEHICLE_COCO_IDS.keys())

    results_gen = model.track(**track_kwargs)

    for r in results_gen:
        frame = r.orig_img.copy()
        boxes = r.boxes

        if boxes is not None and boxes.id is not None:
            ids = boxes.id.cpu().numpy().astype(int)
            xyxys = boxes.xyxy.cpu().numpy()
            clss = boxes.cls.cpu().numpy().astype(int)

            for tid, xyxy, c in zip(ids, xyxys, clss):
                cname = class_names[int(c)]

                x1, y1, x2, y2 = map(int, xyxy)
                box_h = y2 - y1
                if not is_plate_model and box_h < MIN_VEHICLE_BOX_HEIGHT_FRAC * frame_h:
                    continue  # too small / too far away — skip entirely, no box, no log entry

                plate_box = get_plate_box(xyxy, cname, frame.shape)
                if plate_box is None:
                    continue

                entry = track_data.setdefault(tid, {
                    "track_id": int(tid), "class": cname,
                    "first_frame": frame_idx, "last_frame": frame_idx,
                    "best_text": "", "best_conf": 0.0, "hits": 0,
                })
                entry["last_frame"] = frame_idx
                entry["hits"] += 1

                if frame_idx % max(int(ocr_interval), 1) == 0 or entry["best_text"] == "":
                    crop = crop_region(frame, plate_box)
                    text, conf = read_plate(crop)
                    if conf > entry["best_conf"]:
                        entry["best_conf"], entry["best_text"] = conf, text

                px1, py1, px2, py2 = plate_box
                label = entry["best_text"] if entry["best_text"] else f"{cname}?"
                color = (0, 200, 0) if entry["best_text"] else (0, 140, 255)
                cv2.rectangle(frame, (px1, py1), (px2, py2), color, 2)
                cv2.putText(frame, f"ID{tid} {label}", (px1, max(py1 - 8, 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

        writer.write(frame)
        frame_idx += 1
        if total_frames:
            progress(min(frame_idx / total_frames, 0.98), desc=f"Frame {frame_idx}/{total_frames}")

    writer.release()

    if frame_idx == 0:
        raise RuntimeError("No frames were read from the uploaded video — the file may be corrupt or in an unsupported container.")

    progress(0.99, desc="Encoding for browser playback...")
    final_out = f"{HOME}/anpr_output.mp4"
    if os.path.exists(final_out):
        os.remove(final_out)
    os.system(f'ffmpeg -y -i "{tmp_out}" -vcodec libx264 -pix_fmt yuv420p -loglevel error "{final_out}"')
    if not os.path.exists(final_out):  # ffmpeg missing/failed -> fall back to the raw file
        final_out = tmp_out

    rows = []
    for tid, d in track_data.items():
        rows.append({
            "track_id": tid,
            "vehicle_class": d["class"],
            "plate_number": d["best_text"] or "NOT_READ",
            "confidence": round(d["best_conf"], 2),
            "first_seen_s": round(d["first_frame"] / fps, 2),
            "last_seen_s": round(d["last_frame"] / fps, 2),
        })
    rows.sort(key=lambda r: r["first_seen_s"])

    txt_path = f"{HOME}/detected_plates.txt"
    with open(txt_path, "w") as f:
        f.write(f"ANPR Report - {len(rows)} vehicle(s) detected\n")
        f.write("=" * 60 + "\n\n")
        for r in rows:
            f.write(
                f"Vehicle #{r['track_id']:<4} | Plate: {r['plate_number']:<12} | "
                f"Confidence: {r['confidence']:<5} | Class: {r['vehicle_class']:<10} | "
                f"Seen: {r['first_seen_s']}s - {r['last_seen_s']}s\n"
            )
        f.write("\n" + "=" * 60 + "\n")
        plates_only = [r["plate_number"] for r in rows if r["plate_number"] != "NOT_READ"]
        f.write(f"Unique plate numbers read ({len(plates_only)}):\n")
        for p in plates_only:
            f.write(f"  - {p}\n")

    with open(txt_path) as f:
        report_text = f.read()

    progress(1.0, desc="Done!")
    return final_out, report_text, txt_path


def process_video(video_path, progress=None):
    """Any internal failure is turned into a readable error message
    instead of an unhandled exception."""
    if progress is None:
        progress = lambda frac, desc="": None
    if video_path is None:
        raise ValueError("Please provide a video path.")
    try:
        return _process_video_impl(video_path, progress)
    except Exception as e:
        import traceback
        traceback.print_exc()  # full traceback printed to the notebook cell output
        raise RuntimeError(f"Processing failed: {type(e).__name__}: {e}") from e

Warm up EasyOCR's models once now (downloads them the first time), so the pipeline call below isn't stuck waiting on a slow one-time download.

In [ ]:
print("Pre-downloading EasyOCR models (only happens once per runtime)...")
get_ocr_reader()
print("✅ OCR models ready")

## 5. Upload a test video

Colab's built-in file picker — a plain upload with no browser video-preview step and no external networking, so there's nothing here that can produce a "Failed to fetch".

In [ ]:
from google.colab import files

print("Choose a short video file to upload (a few seconds is plenty for a quick test):")
uploaded = files.upload()
VIDEO_PATH = list(uploaded.keys())[0]
print("Uploaded:", VIDEO_PATH)

## 6. Run the pipeline

Runs detection + tracking + OCR end to end, prints progress every 10%, then shows the annotated video inline and prints the plate report.

In [ ]:
class SimpleProgress:
    """Plain print-based progress reporter."""
    def __init__(self):
        self._last = -1

    def __call__(self, frac, desc=""):
        pct = int(frac * 100)
        if pct != self._last and pct % 10 == 0:
            print(f"[{pct:3d}%] {desc}")
            self._last = pct


output_video_path, report_text, txt_path = process_video(VIDEO_PATH, progress=SimpleProgress())

print("\n" + "=" * 60)
print("DONE")
print("=" * 60)
print(f"Processed video: {output_video_path}")
print(f"Report file:     {txt_path}\n")
print(report_text)

## 7. Play the processed video inline

In [ ]:
from IPython.display import HTML
from base64 import b64encode

mp4_bytes = open(output_video_path, "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4_bytes).decode()

HTML(f"""
<video width="640" controls>
  <source src="{data_url}" type="video/mp4">
</video>
""")

## Troubleshooting

- **`ImportError` on `ultralytics`/`easyocr`/`cv2`** → the install cell (Section 1) may not have finished, or the runtime already had conflicting versions loaded. `Runtime → Restart session`, then run all cells again from the top (packages are already installed, this just reloads Python cleanly).
- **"Processing failed: ..." after running Section 6** → the full Python traceback is printed right above that message in the same cell's output — that tells you exactly what broke.
- **"No GPU found"** → `Runtime → Change runtime type → T4 GPU`, then re-run from the top.
- **The inline video in Section 7 doesn't play** → make sure the `ffmpeg` apt-get cell (Section 1) ran without errors; the pipeline falls back to the raw `mp4v` file if the H.264 re-encode fails, and some browsers won't preview that format inline (though it will still download and play fine locally — check `output_video_path`).
- **Plate text is `NOT_READ` for most cars** → you're on the generic `yolov9c.pt` fallback (no custom model). Run Section 2 to train a real plate detector on the Roboflow dataset, or manually set `DEFAULT_WEIGHTS` (Section 3) to the path of your own `best.pt`.
- **Processing is slow** → raise `OCR_EVERY_N_FRAMES` in the pipeline cell (Section 4) before running Section 6, or lower the video's resolution before upload.
- **The upload widget in Section 5 doesn't appear / hangs** → this is Colab's own native file picker (`google.colab.files.upload()`), not a custom component — a page refresh and re-running just that cell usually fixes a stuck picker.